# Качество воды в местах для купания — EDA

**Домен:** Supluskohad (supluskohad_uuringud)  
**Источник:** Terviseamet — vtiav.sm.ee  
**Цель:** Первичное знакомство с данными: структура, распределение, аномалии.

---
## Вопросы, которые хотим ответить:
1. Сколько проб, за какой период?
2. Какой % нарушений?
3. Какие параметры чаще всего нарушаются?
4. Есть ли сезонность? Географические различия?
5. Какие признаки могут быть полезны для модели?

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_domain
from features import add_time_features

# Стиль графиков
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Библиотеки загружены ✓')

## 1. Загрузка данных

In [ ]:
df = load_domain('supluskoha')
print(f'Загружено проб: {len(df)}')
print(f'Колонки: {df.columns.tolist()}')

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Пропущенные значения

In [ ]:
missing = df.isnull().sum() / len(df) * 100
missing_df = missing[missing > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
missing_df.plot(kind='bar', ax=ax, color='salmon')
ax.set_title('Доля пропущенных значений по признакам (%)')
ax.set_ylabel('% пропусков')
ax.axhline(y=50, color='red', linestyle='--', label='50% порог')
ax.legend()
plt.tight_layout()
plt.show()

print(missing_df)

## 3. Целевая переменная: баланс классов

In [ ]:
from evaluate import plot_class_distribution

df_labeled = df.dropna(subset=['compliant'])
print(f'Проб с известным статусом: {len(df_labeled)} из {len(df)}')
print(f'\nРаспределение:')
print(df_labeled['compliant'].value_counts())
print(f'\n% нарушений: {(df_labeled["compliant"] == 0).mean() * 100:.1f}%')

plot_class_distribution(df_labeled['compliant'], title='Баланс классов: Supluskohad')

## 4. Временная динамика

In [ ]:
df_time = add_time_features(df_labeled)

# Нарушения по годам
yearly = df_time.groupby('year')['compliant'].agg(
    total='count',
    violations=lambda x: (x == 0).sum()
)
yearly['violation_rate'] = yearly['violations'] / yearly['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

yearly['total'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Количество проб по годам')
axes[0].set_xlabel('Год')

yearly['violation_rate'].plot(kind='line', marker='o', ax=axes[1], color='salmon')
axes[1].set_title('Доля нарушений по годам (%)')
axes[1].set_xlabel('Год')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

In [ ]:
# Сезонность (купальный сезон = лето)
monthly = df_time.groupby('month')['compliant'].agg(
    total='count',
    violations=lambda x: (x == 0).sum()
)
monthly['violation_rate'] = monthly['violations'] / monthly['total'] * 100

fig, ax = plt.subplots(figsize=(8, 4))
monthly['violation_rate'].plot(kind='bar', ax=ax, color=['salmon' if v > monthly['violation_rate'].mean() else 'steelblue' 
                                                           for v in monthly['violation_rate']])
ax.set_title('Доля нарушений по месяцам (%)')
ax.set_xlabel('Месяц')
ax.set_ylabel('%')
ax.axhline(y=monthly['violation_rate'].mean(), color='red', linestyle='--', label='Среднее')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Географический анализ

In [ ]:
# Нарушения по уездам
county_stats = df_labeled.groupby('county')['compliant'].agg(
    total='count',
    violations=lambda x: (x == 0).sum()
).sort_values('violations', ascending=False)
county_stats['violation_rate'] = county_stats['violations'] / county_stats['total'] * 100

fig, ax = plt.subplots(figsize=(10, 5))
county_stats['violation_rate'].sort_values().plot(kind='barh', ax=ax, color='salmon')
ax.set_title('Доля нарушений по уездам (%)')
ax.set_xlabel('%')
ax.axvline(x=county_stats['violation_rate'].mean(), color='red', linestyle='--', label='Среднее')
ax.legend()
plt.tight_layout()
plt.show()

print('Топ-5 уездов по количеству нарушений:')
print(county_stats.head())

## 6. Распределение числовых параметров

In [ ]:
# Числовые параметры для supluskohad
numeric_cols = ['e_coli', 'enterococci', 'ph', 'transparency']
available_cols = [c for c in numeric_cols if c in df_labeled.columns]

fig, axes = plt.subplots(2, len(available_cols), figsize=(4*len(available_cols), 8))

for i, col in enumerate(available_cols):
    # Гистограмма
    df_labeled[col].dropna().plot(kind='hist', bins=30, ax=axes[0][i], color='steelblue', edgecolor='white')
    axes[0][i].set_title(f'{col}: распределение')
    
    # Box plot: норма vs нарушение
    df_labeled.boxplot(column=col, by='compliant', ax=axes[1][i])
    axes[1][i].set_title(f'{col}: норма vs нарушение')
    axes[1][i].set_xlabel('compliant (1=норма, 0=нарушение)')

plt.tight_layout()
plt.show()

## 7. Корреляционная матрица

In [ ]:
cols_for_corr = available_cols + ['compliant']
corr = df_labeled[cols_for_corr].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax)
ax.set_title('Корреляционная матрица (supluskohad)')
plt.tight_layout()
plt.show()

## 8. Итог EDA: ключевые инсайты

Заполнить после запуска ноутбука:

1. **Объём данных:** ... проб, ... нарушений (...)%
2. **Баланс классов:** сбалансированный / несбалансированный (соотношение ...)
3. **Главный нарушитель:** ... (параметр с наибольшим числом превышений)
4. **Сезонность:** ...
5. **Проблемные уезды:** ...
6. **Пропуски:** критичные признаки с большим % пропусков: ...
7. **Для модели:** наиболее перспективные признаки: ...